## v3.1 — GLMM/target encoding + XGBoost + LightGBM + CatBoost + HistGB, tuned, ensembled

Extends the v1/v2 line with the two levers the source-competition research
actually validated (see `docs/Nishkarsh/feature_engineering.md`): **target/GLMM
encoding** for categoricals instead of one-hot/native-categorical, and a
**LightGBM/CatBoost** comparison alongside XGBoost. Adds real hyperparameter
tuning (random search) for every model, then blends their predictions.

**Deliberately not used:** `customer_id`, `last_name`. Both are reused
quasi-identifiers in this dataset (92%/99.5% of test rows share one with some
train row, ~80% same-target purity within a shared value) — encoding either
one is leakage that happens to look like a normal feature, not a real signal.
Full writeup in this conversation; not repeated here.

### Scoring convention (confirmed with the organizers, not assumed)

The competition's F1 grader scores with `pos_label=0` (non-churn / "stays" as
the positive class) rather than the natural `pos_label=1` (churn) reading of
this task -- confirmed directly by an organizer. This is *not* a modeling
problem: the exact same trained models and probability outputs score
~0.66 F1 under the churn-positive convention and ~0.92 F1 under the
non-churn-positive one, because F1's precision/recall balance is
fundamentally different depending on which class counts as "positive," and
the majority class (79% of rows) is much easier to get right. Every
threshold search and every hyperparameter search in this notebook targets
`pos_label=0` throughout, since that's what's actually graded -- this
matters for hyperparameter selection too, not just the final cutoff, since
the best-scoring config under one convention isn't guaranteed to be the
best-scoring config under the other. (A quick check found training-time
class balancing barely affects the achievable ceiling either way -- 0.9158
vs 0.9159 -- so `sample_weight='balanced'` is kept for the minor stability
it gives the ranking, not because it's load-bearing for the final score.)

Earlier versions (v1.1-v2.2, and the first pass of this notebook) all
optimized for `pos_label=1` and are **not directly comparable** to the
numbers below -- their reported F1s were measuring a different, ungraded
target the whole time.

### GPU note (confirmed empirically, not assumed)
- **CatBoost → GPU** (`task_type='GPU'`). Verified with `nvidia-smi` during a
  stress test: real GPU utilization, 5.68 / 6.14 GB used — confirms the VRAM
  ceiling is real, not theoretical.
- **XGBoost → CPU.** This install's `xgboost.build_info()` reports
  `USE_CUDA: False` — a genuine CPU-only build, not a config choice.
- **LightGBM → CPU.** The standard Windows pip wheel has no GPU support built in.

Only one model (CatBoost) ever touches the GPU, and every model here trains
in an ordinary sequential notebook cell — nothing is parallelized across
models — so there's never more than one model's memory footprint live at a
time. `gc.collect()` is called explicitly after each model's block anyway, to
make that sequencing explicit rather than incidental.


In [1]:
import gc
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.preprocessing import TargetEncoder
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

RANDOM_STATE = 42
rng = np.random.RandomState(RANDOM_STATE)

# Confirmed with an organizer: the grader computes F1 with the non-churn (0)
# class as positive, not churn (1). Every threshold/hyperparameter search
# below targets this directly.
POS_LABEL = 0


In [2]:
train = pd.read_csv('../../data/train.csv')
test = pd.read_csv('../../data/test.csv')
print("train:", train.shape, "test:", test.shape)


train: (90000, 14) test: (30000, 13)


In [3]:
ID_COLS = ['id', 'customer_id', 'last_name']
TARGET = 'exit_status'
PROD_COUNT_IMPUTE_FEATURES = [
    'age', 'is_active', 'acc_balance', 'country', 'credit_score',
    'has_card', 'estimated_salary', 'tenure',
]
NUM_COLS = ['credit_score', 'age', 'tenure', 'acc_balance', 'has_card', 'is_active', 'estimated_salary']
CAT_COLS = ['country', 'gender', 'prod_count']


def make_X(df):
    return df.drop(columns=[c for c in ID_COLS + [TARGET] if c in df.columns])


### `BankChurnImputer`

Unchanged from v1.2/v2.2 -- fills `country`->`"Unknown"`, `acc_balance`->
median-by-country, `credit_score`->median, `prod_count`->`RandomForestClassifier`
prediction, all fit only on training-fold data. Full imputation (rather than
v1.1's "leave country/credit_score raw" approach) is the right call here
specifically because target encoding needs a concrete category to encode --
there's no such thing as a native-NaN-aware target encoding.


In [4]:
class BankChurnImputer(BaseEstimator, TransformerMixin):
    def __init__(self, prod_count_features=PROD_COUNT_IMPUTE_FEATURES, random_state=RANDOM_STATE):
        self.prod_count_features = prod_count_features
        self.random_state = random_state

    def fit(self, X, y=None):
        X = X.copy()
        X['country'] = X['country'].fillna('Unknown')
        self.balance_median_by_country_ = X.groupby('country')['acc_balance'].median()
        self.balance_global_median_ = X['acc_balance'].median()
        self.credit_score_median_ = X['credit_score'].median()

        X['acc_balance'] = X['acc_balance'].fillna(X['country'].map(self.balance_median_by_country_))
        X['acc_balance'] = X['acc_balance'].fillna(self.balance_global_median_)
        X['credit_score'] = X['credit_score'].fillna(self.credit_score_median_)

        known = X.dropna(subset=['prod_count'])
        Xk = pd.get_dummies(known[self.prod_count_features], columns=['country'])
        self.prod_count_columns_ = Xk.columns
        yk = known['prod_count'].astype(int)

        self.prod_count_model_ = RandomForestClassifier(
            n_estimators=300, max_depth=None, min_samples_leaf=5,
            random_state=self.random_state, n_jobs=-1,
        )
        self.prod_count_model_.fit(Xk, yk)
        return self

    def transform(self, X):
        X = X.copy()
        X['country'] = X['country'].fillna('Unknown')
        X['acc_balance'] = X['acc_balance'].fillna(X['country'].map(self.balance_median_by_country_))
        X['acc_balance'] = X['acc_balance'].fillna(self.balance_global_median_)
        X['credit_score'] = X['credit_score'].fillna(self.credit_score_median_)

        missing = X['prod_count'].isna()
        if missing.any():
            Xm = pd.get_dummies(X.loc[missing, self.prod_count_features], columns=['country'])
            Xm = Xm.reindex(columns=self.prod_count_columns_, fill_value=0)
            X.loc[missing, 'prod_count'] = self.prod_count_model_.predict(Xm)
        return X


### Preprocessed, cached CV folds

Imputation and target encoding don't depend on model hyperparameters, so
redoing them for every random-search trial would be pure waste (dozens of
trials x 4 models). Instead, preprocessing runs **once** per fold layout --
a cheap 3-fold split for the hyperparameter search phase, a proper 5-fold
split for final scoring -- and every model's search and final evaluation
reuses the same cached arrays.

Target encoding uses `sklearn.preprocessing.TargetEncoder`: cross-fitted
(`fit_transform`) on the training portion of each fold so a row's own label
never leaks into its own encoded value, then applied (`transform`, no further
fitting) to that fold's held-out rows.


In [5]:
def build_preprocessed_folds(X, y, n_splits, random_state=RANDOM_STATE):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    folds = []
    for tr_idx, va_idx in skf.split(X, y):
        X_tr, X_va = X.iloc[tr_idx].copy(), X.iloc[va_idx].copy()
        y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]

        imp = BankChurnImputer()
        imp.fit(X_tr)
        X_tr, X_va = imp.transform(X_tr), imp.transform(X_va)

        te = TargetEncoder(target_type='binary', random_state=random_state, cv=5)
        X_tr_cat = te.fit_transform(X_tr[CAT_COLS], y_tr)
        X_va_cat = te.transform(X_va[CAT_COLS])

        X_tr_final = np.hstack([X_tr[NUM_COLS].to_numpy(), X_tr_cat])
        X_va_final = np.hstack([X_va[NUM_COLS].to_numpy(), X_va_cat])

        folds.append((X_tr_final, y_tr.to_numpy(), X_va_final, y_va.to_numpy(), va_idx))
    return folds


X = make_X(train)
y = train[TARGET]

print("Building 3-fold cache (hyperparameter search)...")
search_folds = build_preprocessed_folds(X, y, n_splits=3)
print("Building 5-fold cache (final evaluation)...")
final_folds = build_preprocessed_folds(X, y, n_splits=5)
print("Done.")


Building 3-fold cache (hyperparameter search)...


Building 5-fold cache (final evaluation)...


Done.


### Evaluation + random search harness

Same F1-threshold-search principle as v1/v2, now operating on the cached
numpy folds so both the search phase and the final phase can reuse it.


In [6]:
def evaluate_folds(build_model_fn, folds, n_total):
    oof = np.zeros(n_total)
    for X_tr, y_tr, X_va, y_va, va_idx in folds:
        model = build_model_fn()
        sw = compute_sample_weight('balanced', y_tr)
        model.fit(X_tr, y_tr, sample_weight=sw)
        oof[va_idx] = model.predict_proba(X_va)[:, 1]

    thresholds = np.linspace(0.02, 0.98, 97)
    f1s = [f1_score(y.to_numpy(), oof > t, pos_label=POS_LABEL) for t in thresholds]
    best = int(np.argmax(f1s))
    return oof, thresholds[best], f1s[best]


def random_search(build_model_fn_factory, param_space, folds, n_total, n_iter=12, random_state=RANDOM_STATE):
    local_rng = np.random.RandomState(random_state)
    rows = []
    best = {'f1': -1.0, 'params': None}
    for i in range(n_iter):
        params = {k: v[local_rng.randint(len(v))] for k, v in param_space.items()}
        _, thr, f1 = evaluate_folds(lambda p=params: build_model_fn_factory(**p), folds, n_total)
        rows.append({**params, 'oof_f1': f1, 'threshold': thr})
        if f1 > best['f1']:
            best = {'f1': f1, 'params': params, 'threshold': thr}
    return best, pd.DataFrame(rows).sort_values('oof_f1', ascending=False).reset_index(drop=True)


### XGBoost (CPU) -- tuned


In [7]:
xgb_param_space = {
    'n_estimators': [200, 300, 400, 600],
    'max_depth': [3, 4, 5, 6, 7],
    'learning_rate': [0.02, 0.03, 0.05, 0.08, 0.1],
    'subsample': [0.6, 0.7, 0.8, 0.9, 1.0],
    'colsample_bytree': [0.6, 0.7, 0.8, 0.9, 1.0],
    'min_child_weight': [1, 3, 5, 7],
    'reg_lambda': [0.5, 1.0, 2.0, 5.0],
}

def build_xgb(**params):
    return XGBClassifier(
        **params, tree_method='hist', random_state=RANDOM_STATE, verbosity=0, n_jobs=-1,
    )

print("Searching XGBoost hyperparameters (3-fold)...")
best_xgb, xgb_search_results = random_search(build_xgb, xgb_param_space, search_folds, len(X), n_iter=12)
print("Best params:", best_xgb['params'])

print("Final 5-fold evaluation with best params...")
oof_xgb, thr_xgb, f1_xgb = evaluate_folds(lambda: build_xgb(**best_xgb['params']), final_folds, len(X))
print(f"XGBoost tuned: OOF F1={f1_xgb:.4f}  best_threshold={thr_xgb:.2f}")

gc.collect()


Searching XGBoost hyperparameters (3-fold)...


Best params: {'n_estimators': 300, 'max_depth': 5, 'learning_rate': 0.05, 'subsample': 0.8, 'colsample_bytree': 1.0, 'min_child_weight': 7, 'reg_lambda': 5.0}
Final 5-fold evaluation with best params...


XGBoost tuned: OOF F1=0.9160  best_threshold=0.82


1142

### LightGBM (CPU) -- tuned


In [8]:
lgb_param_space = {
    'n_estimators': [200, 300, 400, 600],
    'num_leaves': [15, 31, 63, 127],
    'max_depth': [-1, 4, 6, 8],
    'learning_rate': [0.02, 0.03, 0.05, 0.08, 0.1],
    'subsample': [0.6, 0.7, 0.8, 0.9, 1.0],
    'colsample_bytree': [0.6, 0.7, 0.8, 0.9, 1.0],
    'min_child_samples': [5, 10, 20, 30],
    'reg_lambda': [0.0, 0.5, 1.0, 2.0, 5.0],
}

def build_lgb(**params):
    return LGBMClassifier(
        **params, random_state=RANDOM_STATE, verbose=-1, n_jobs=-1,
    )

print("Searching LightGBM hyperparameters (3-fold)...")
best_lgb, lgb_search_results = random_search(build_lgb, lgb_param_space, search_folds, len(X), n_iter=12)
print("Best params:", best_lgb['params'])

print("Final 5-fold evaluation with best params...")
oof_lgb, thr_lgb, f1_lgb = evaluate_folds(lambda: build_lgb(**best_lgb['params']), final_folds, len(X))
print(f"LightGBM tuned: OOF F1={f1_lgb:.4f}  best_threshold={thr_lgb:.2f}")

gc.collect()


Searching LightGBM hyperparameters (3-fold)...


Best params: {'n_estimators': 600, 'num_leaves': 31, 'max_depth': -1, 'learning_rate': 0.03, 'subsample': 0.7, 'colsample_bytree': 0.9, 'min_child_samples': 10, 'reg_lambda': 0.5}
Final 5-fold evaluation with best params...


LightGBM tuned: OOF F1=0.9159  best_threshold=0.81


772

### CatBoost (GPU) -- tuned

The only model in this notebook touching the GPU. Its search space is kept
well below the levels used in the earlier VRAM stress test (depth <=8,
iterations <=800 here vs. depth 10 / 2000 iterations / 300k rows in the test
that used 5.68 GB), so a full run comfortably fits in 6 GB.


In [9]:
cat_param_space = {
    'iterations': [200, 300, 400, 600, 800],
    'depth': [4, 5, 6, 7, 8],
    'learning_rate': [0.02, 0.03, 0.05, 0.08, 0.1],
    'l2_leaf_reg': [1.0, 3.0, 5.0, 7.0, 9.0],
}

def build_cat(**params):
    return CatBoostClassifier(
        **params, task_type='GPU', devices='0', random_seed=RANDOM_STATE, verbose=False,
    )

print("Searching CatBoost hyperparameters (3-fold, GPU)...")
best_cat, cat_search_results = random_search(build_cat, cat_param_space, search_folds, len(X), n_iter=12)
print("Best params:", best_cat['params'])

print("Final 5-fold evaluation with best params...")
oof_cat, thr_cat, f1_cat = evaluate_folds(lambda: build_cat(**best_cat['params']), final_folds, len(X))
print(f"CatBoost tuned: OOF F1={f1_cat:.4f}  best_threshold={thr_cat:.2f}")

gc.collect()


Searching CatBoost hyperparameters (3-fold, GPU)...


Best params: {'iterations': 400, 'depth': 8, 'learning_rate': 0.02, 'l2_leaf_reg': 3.0}
Final 5-fold evaluation with best params...


CatBoost tuned: OOF F1=0.9160  best_threshold=0.80


0

### HistGradientBoostingClassifier (CPU) -- tuned

Free to include (sklearn built-in, no GPU/extra dependency), and adds a
fourth, differently-biased model for the ensemble.


In [10]:
hgb_param_space = {
    'max_iter': [200, 300, 400, 600],
    'max_depth': [4, 5, 6, 7, None],
    'learning_rate': [0.02, 0.03, 0.05, 0.08, 0.1],
    'l2_regularization': [0.0, 0.5, 1.0, 2.0],
    'max_leaf_nodes': [15, 31, 63, 127],
}

def build_hgb(**params):
    return HistGradientBoostingClassifier(**params, random_state=RANDOM_STATE)

print("Searching HistGB hyperparameters (3-fold)...")
best_hgb, hgb_search_results = random_search(build_hgb, hgb_param_space, search_folds, len(X), n_iter=10)
print("Best params:", best_hgb['params'])

print("Final 5-fold evaluation with best params...")
oof_hgb, thr_hgb, f1_hgb = evaluate_folds(lambda: build_hgb(**best_hgb['params']), final_folds, len(X))
print(f"HistGB tuned: OOF F1={f1_hgb:.4f}  best_threshold={thr_hgb:.2f}")

gc.collect()


Searching HistGB hyperparameters (3-fold)...


Best params: {'max_iter': 600, 'max_depth': 4, 'learning_rate': 0.05, 'l2_regularization': 0.0, 'max_leaf_nodes': 63}
Final 5-fold evaluation with best params...


HistGB tuned: OOF F1=0.9159  best_threshold=0.81


24

### Results (pos_label=0, the actual graded convention)

Not compared against v1.1-v2.2's numbers here -- those were computed under
the wrong convention (`pos_label=1`) and aren't on the same scale as these.


In [11]:
results = pd.DataFrame([
    {'model': 'XGBoost (tuned, target-enc)', 'oof_f1': f1_xgb, 'threshold': thr_xgb},
    {'model': 'LightGBM (tuned, target-enc)', 'oof_f1': f1_lgb, 'threshold': thr_lgb},
    {'model': 'CatBoost (tuned, target-enc, GPU)', 'oof_f1': f1_cat, 'threshold': thr_cat},
    {'model': 'HistGB (tuned, target-enc)', 'oof_f1': f1_hgb, 'threshold': thr_hgb},
]).sort_values('oof_f1', ascending=False).reset_index(drop=True)

print("=== v3.1 tuned models (pos_label=0) ===")
results


=== v3.1 tuned models (pos_label=0) ===


,model,oof_f1,threshold
0,"XGBoost (tuned, target-enc)",0.916040,0.82
1,"CatBoost (tuned, target-enc, GPU)",0.915968,0.80
2,"LightGBM (tuned, target-enc)",0.915928,0.81
3,"HistGB (tuned, target-enc)",0.915923,0.81


### Ensemble

Blends the 4 tuned models' final-fold OOF probabilities. Tries equal-weight
averaging first, then a light random weight search on the *same* OOF vectors
(no retraining -- these are already-computed probabilities, so searching
weights is nearly free) to see if an unequal blend does meaningfully better.


In [12]:
oof_matrix = np.column_stack([oof_xgb, oof_lgb, oof_cat, oof_hgb])
model_names = ['xgb', 'lgb', 'cat', 'hgb']
y_arr = y.to_numpy()

def threshold_search(scores, y_true):
    thresholds = np.linspace(0.02, 0.98, 97)
    f1s = [f1_score(y_true, scores > t, pos_label=POS_LABEL) for t in thresholds]
    best = int(np.argmax(f1s))
    return thresholds[best], f1s[best]

# equal-weight blend
equal_blend = oof_matrix.mean(axis=1)
thr_equal, f1_equal = threshold_search(equal_blend, y_arr)
print(f"Equal-weight blend: OOF F1={f1_equal:.4f}  threshold={thr_equal:.2f}")

# light random weight search on the precomputed OOF matrix
best_weights, best_blend_f1, best_blend_thr = None, f1_equal, thr_equal
for _ in range(300):
    w = rng.dirichlet(np.ones(4))
    blend = oof_matrix @ w
    thr, f1 = threshold_search(blend, y_arr)
    if f1 > best_blend_f1:
        best_blend_f1, best_blend_thr, best_weights = f1, thr, w

if best_weights is None:
    best_weights = np.ones(4) / 4
    print("Weight search did not beat equal-weight; keeping equal weights.")
else:
    print(f"Weighted blend: OOF F1={best_blend_f1:.4f}  threshold={best_blend_thr:.2f}  weights={dict(zip(model_names, best_weights.round(3)))}")


Equal-weight blend: OOF F1=0.9161  threshold=0.81


Weighted blend: OOF F1=0.9163  threshold=0.81  weights={'xgb': np.float64(0.071), 'lgb': np.float64(0.25), 'cat': np.float64(0.655), 'hgb': np.float64(0.024)}


### Final fit + submission

Refits `BankChurnImputer` + `TargetEncoder` on the full training set (no more
CV folds needed), refits each of the 4 tuned models on the full training set,
blends their test-set probabilities with the weights found above, and writes
a submission using the OOF-derived threshold. Models are refit one at a time,
in the same order as above, for the same VRAM reason.


In [13]:
import os

X_train_final = make_X(train)
y_train_final = train[TARGET]
X_test_final = make_X(test)

final_imputer = BankChurnImputer()
final_imputer.fit(X_train_final)
X_train_imp = final_imputer.transform(X_train_final)
X_test_imp = final_imputer.transform(X_test_final)

final_te = TargetEncoder(target_type='binary', random_state=RANDOM_STATE, cv=5)
X_train_cat = final_te.fit_transform(X_train_imp[CAT_COLS], y_train_final)
X_test_cat = final_te.transform(X_test_imp[CAT_COLS])

X_train_arr = np.hstack([X_train_imp[NUM_COLS].to_numpy(), X_train_cat])
X_test_arr = np.hstack([X_test_imp[NUM_COLS].to_numpy(), X_test_cat])

sw_final = compute_sample_weight('balanced', y_train_final)
test_proba = np.zeros((len(X_test_arr), 4))

print("Refitting XGBoost on full train...")
m = build_xgb(**best_xgb['params']); m.fit(X_train_arr, y_train_final, sample_weight=sw_final)
test_proba[:, 0] = m.predict_proba(X_test_arr)[:, 1]
del m; gc.collect()

print("Refitting LightGBM on full train...")
m = build_lgb(**best_lgb['params']); m.fit(X_train_arr, y_train_final, sample_weight=sw_final)
test_proba[:, 1] = m.predict_proba(X_test_arr)[:, 1]
del m; gc.collect()

print("Refitting CatBoost on full train (GPU)...")
m = build_cat(**best_cat['params']); m.fit(X_train_arr, y_train_final, sample_weight=sw_final)
test_proba[:, 2] = m.predict_proba(X_test_arr)[:, 1]
del m; gc.collect()

print("Refitting HistGB on full train...")
m = build_hgb(**best_hgb['params']); m.fit(X_train_arr, y_train_final, sample_weight=sw_final)
test_proba[:, 3] = m.predict_proba(X_test_arr)[:, 1]
del m; gc.collect()

test_blend = test_proba @ best_weights
test_pred = (test_blend > best_blend_thr).astype(int)

os.makedirs('outputs', exist_ok=True)
submission = pd.DataFrame({'id': test['id'], 'exit_status': test_pred})
submission.to_csv('outputs/v3_1_ensemble_submission.csv', index=False)
print(f"Ensemble OOF F1 was {best_blend_f1:.4f} vs best single model {results.iloc[0]['model']} at {results.iloc[0]['oof_f1']:.4f}")
submission.head()


Refitting XGBoost on full train...


Refitting LightGBM on full train...


Refitting CatBoost on full train (GPU)...


Refitting HistGB on full train...


Ensemble OOF F1 was 0.9163 vs best single model XGBoost (tuned, target-enc) at 0.9160


,id,exit_status
0,0,0
1,1,1
2,2,0
3,3,0
4,4,0
